# CNV-LDA 01: measurement QC

Builds a QC'd person × measurement matrix (EHR labs/vitals + physical measurements) for the EUR genomatrix cohort, plus per-group coverage tables to choose the missingness cutoff for the LDA.

1. Cohort: 16p11.2 group (DEL / DUP / NC = non-carrier), sex, PCs
2. BigQuery: per-person median of each measurement (most common unit only, extreme 0.1% of records trimmed)
3. Coverage of each measurement by group
4. Box-Cox transform → regress out age, age², sex, PCs → z-score
5. Save to the bucket

Run with **Kernel → Restart Kernel and Run All Cells**. Printed counts of 1–20 are suppressed (and fractions based on them are blanked), so outputs can be shared.

In [ ]:
BRANCH = "dev/cnv-lda"
REPO = "https://github.com/mollysacks/AoU-utils.git"

from pathlib import Path
BUCKET_DIR = Path.home() / "workspace" / "rw-migration-aou-rw-568aa47b"
GENOMATRIX = BUCKET_DIR / "genomatrix" / "genomatrix_CNV_EUR.20250715.csv"
OUT_DIR = BUCKET_DIR / "cnv-lda" / "01_measurement_qc"

CNV_COL = "16p11.2_BP4_BP5"  # copy number: 1 = DEL, 2 = non-carrier, 3 = DUP
N_PCS = 10
MIN_FRAC = 0.05      # keep measurements present in >= this fraction of the cohort
TRIM_PERMILLE = 1    # drop records outside each measurement's 0.1-99.9th percentiles
REFRESH = False      # re-run the BigQuery pull even if a saved copy exists (set True after changing the settings above)

In [ ]:
%pip install --quiet --user --no-deps --force-reinstall "git+{REPO}@{BRANCH}"

In [ ]:
# Make the fresh install importable without restarting the kernel
import importlib, importlib.metadata, json, site, sys

site.addsitedir(site.getusersitepackages())
importlib.invalidate_caches()
for name in [m for m in sys.modules if m == "aou_utils" or m.startswith("aou_utils.")]:
    del sys.modules[name]

direct_url = json.loads(importlib.metadata.distribution("aou-utils").read_text("direct_url.json"))
print("installed", BRANCH, "@", direct_url["vcs_info"]["commit_id"][:7])

In [ ]:
import numpy as np
import pandas as pd

from aou_utils import measurements, privacy

OUT_DIR.mkdir(parents=True, exist_ok=True)
GROUPS = ["DEL", "DUP", "NC"]

## 1. Cohort

In [ ]:
pcs = [f"PC{i}" for i in range(1, N_PCS + 1)]
header = pd.read_csv(GENOMATRIX, nrows=0).columns
if CNV_COL not in header:
    raise KeyError(f"{CNV_COL!r} not in genomatrix; 16p columns are {[c for c in header if c.startswith('16p')]}")

cohort = pd.read_csv(GENOMATRIX, usecols=["person_id", "sex_at_birth", CNV_COL, *pcs]).set_index("person_id")
assert cohort.index.is_unique
cohort["group"] = cohort[CNV_COL].map({1: "DEL", 2: "NC", 3: "DUP"})
cohort["male"] = cohort["sex_at_birth"].map({"Male": 1.0, "Female": 0.0})  # anything else -> NaN, excluded from adjustment

print("dropped, CNV not 1/2/3:", privacy.suppress(pd.Series([cohort['group'].isna().sum()]))[0])
cohort = cohort.dropna(subset=["group"])
print("no Male/Female sex at birth:", privacy.suppress(pd.Series([cohort['male'].isna().sum()]))[0])
privacy.suppress(cohort["group"].value_counts())

## 2. Per-person medians from BigQuery

This is the slow step. The result is saved, and later runs reload it unless `REFRESH = True`.

In [ ]:
medians_path = OUT_DIR / "person_medians.parquet"
if medians_path.exists() and not REFRESH:
    medians = pd.read_parquet(medians_path)
else:
    medians = measurements.person_medians(
        cohort.index, min_people=int(MIN_FRAC * len(cohort)), trim_permille=TRIM_PERMILLE
    )
    medians.to_parquet(medians_path)
print(f"{medians['concept_id'].nunique()} measurements, {len(medians):,} person-level medians")

## 3. Coverage by group

`frac_carrier_min` is the lower of the DEL and DUP coverage. LDA needs complete data, so the threshold table below shows how many people in each group have *all* measurements at each cutoff.

In [ ]:
names = measurements.concept_names(pd.concat([medians["concept_id"], medians["unit_concept_id"]]))
names = names.set_index("concept_id")

concepts = medians.groupby("concept_id").agg(
    unit_concept_id=("unit_concept_id", "first"),
    n_people=("person_id", "size"),
    n_records=("n_records", "sum"),
    n_pm_records=("n_pm_records", "sum"),
)
concepts["name"] = names["concept_name"].reindex(concepts.index).to_numpy()
concepts["vocabulary"] = names["vocabulary_id"].reindex(concepts.index).to_numpy()
concepts["unit"] = names["concept_name"].reindex(concepts["unit_concept_id"]).to_numpy()
concepts["pm_fraction"] = concepts["n_pm_records"] / concepts["n_records"]

group_of_row = cohort["group"].reindex(medians["person_id"]).to_numpy()
counts = medians.assign(group=group_of_row).groupby(["concept_id", "group"]).size().unstack(fill_value=0)
counts = counts.reindex(columns=GROUPS, fill_value=0)
coverage = counts / cohort["group"].value_counts()[GROUPS]
concepts = concepts.join(counts.add_prefix("n_")).join(coverage.add_prefix("frac_"))
concepts["frac_carrier_min"] = coverage[["DEL", "DUP"]].min(axis=1)
concepts = concepts.sort_values("frac_carrier_min", ascending=False)

safe = concepts[["name", "unit", "pm_fraction"]].copy()
safe["n_people"] = privacy.suppress(concepts["n_people"])
for g in GROUPS:
    safe[f"frac_{g}"] = privacy.mask_fractions(concepts[f"frac_{g}"], concepts[f"n_{g}"]).round(3)
pd.set_option("display.max_rows", 100, "display.max_colwidth", 60)
safe.head(100)

In [ ]:
present = measurements.to_wide(medians).reindex(cohort.index).notna()
rows = []
for t in [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    keep = concepts.index[concepts["frac_carrier_min"] >= t]
    complete = present[keep].all(axis=1)
    n_complete = cohort.loc[complete, "group"].value_counts().reindex(GROUPS, fill_value=0)
    rows.append({"carrier_coverage_cutoff": t, "n_measurements": len(keep), **n_complete.add_prefix("complete_")})
thresholds = pd.DataFrame(rows).set_index("carrier_coverage_cutoff")
for g in GROUPS:
    thresholds[f"complete_{g}"] = privacy.suppress(thresholds[f"complete_{g}"])
thresholds

## 4. Box-Cox, covariate adjustment

Each measurement is Box-Cox transformed (shifted first if it has values ≤ 0), then age, age², sex and PCs are regressed out, using each person's median age across the records behind that measurement. Residuals are z-scored.

In [ ]:
values = measurements.to_wide(medians, "value").reindex(cohort.index)
ages = measurements.to_wide(medians, "age").reindex(cohort.index)

transformed, boxcox_params = measurements.boxcox(values)
print("left out (< 3 distinct values):", [concepts.loc[c, 'name'] for c in values.columns.difference(transformed.columns)])

qc = measurements.residualize(transformed, ages[transformed.columns], cohort[["male", *pcs]])
concepts = concepts.join(boxcox_params)
concepts["n_beyond_5sd"] = (qc.abs() > 5).sum().reindex(concepts.index)

extreme = concepts.sort_values("n_beyond_5sd", ascending=False).head(20)
pd.DataFrame({"name": extreme["name"], "lambda": extreme["lambda"].round(2),
              "n_beyond_5sd": privacy.suppress(extreme["n_beyond_5sd"])})

## 5. Save

- `person_medians.parquet`: long per-person medians from BigQuery (raw values, saved in step 2)
- `measurements_qc.parquet`: person × measurement matrix after Box-Cox and covariate adjustment (columns are concept ids)
- `measurement_concepts.parquet`: per-measurement names, units, coverage, Box-Cox lambda/shift
- `cohort.parquet`: group, sex, PCs

In [ ]:
qc.columns = qc.columns.astype(str)  # parquet needs string column names
qc.to_parquet(OUT_DIR / "measurements_qc.parquet")
concepts.to_parquet(OUT_DIR / "measurement_concepts.parquet")
cohort.to_parquet(OUT_DIR / "cohort.parquet")
print("saved to", OUT_DIR)